# 01 — Embeddings & Similarity: Mencari Berdasarkan Makna

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/07-embeddings-vectordb/01_embeddings_dan_similarity.ipynb)

**Embedding** adalah vektor angka yang mewakili *makna* sebuah objek (kata, kalimat, gambar). Objek yang maknanya mirip mendapat vektor yang **berdekatan**. Kita sebenarnya sudah bertemu embedding beberapa kali:
- fitur CNN sebelum layer klasifikasi (modul 02),
- token CLS di ViT dan BERT (modul 05–06),
- embedding gambar dan teks CLIP yang bisa saling dicari (modul 05).

Notebook ini fokus pada **embedding kalimat**, fondasi *semantic search* dan **RAG** (modul 09):
1. Ukuran kemiripan: cosine, dot product, dan euclidean.
2. Model **sentence-transformers** dan cara kerjanya (mean pooling).
3. **Semantic search** vs pencarian kata kunci (TF-IDF).
4. Mengukur kualitas pencarian dengan **recall@k** dan **MRR**, termasuk pencarian **lintas bahasa**.

**Data:** NusaX-Senti ([IndoNLP/nusax](https://github.com/IndoNLP/nusax), lisensi **CC-BY-SA**), 1.000 kalimat ulasan/komentar. NusaX adalah dataset **paralel**: kalimat dengan `id` yang sama tersedia dalam bahasa Indonesia **dan** terjemahan bahasa Inggrisnya. Sifat ini kita manfaatkan untuk mengukur kualitas pencarian secara objektif.

**Model:** [`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) (Apache-2.0): model embedding kalimat **multibahasa** (50+ bahasa, termasuk Indonesia), 118 juta parameter, vektor 384 dimensi.

## 0. Cek GPU

Menghitung embedding jauh lebih cepat di GPU (**Runtime → Change runtime type → T4 GPU**). Untuk 1.000 kalimat di notebook ini, CPU pun masih cukup.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, berjalan di CPU (lebih lambat, tetapi cukup untuk notebook ini).")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Instalasi dan Data

`sentence-transformers` (Apache-2.0) adalah library standar untuk model embedding kalimat.

In [ ]:
%pip install -q sentence-transformers scikit-learn pandas

Muat NusaX bahasa Indonesia dan Inggris, lalu gabungkan berdasarkan `id` sehingga setiap baris berisi satu pasangan terjemahan.

In [ ]:
import numpy as np
import pandas as pd

NUSAX_URL = "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/sentiment"

def muat(bahasa):
    return pd.concat([pd.read_csv(f"{NUSAX_URL}/{bahasa}/{s}.csv") for s in ["train", "valid", "test"]])

df = muat("indonesian").merge(muat("english")[["id", "text"]], on="id", suffixes=("", "_en")).reset_index(drop=True)
print(f"{len(df)} pasangan kalimat | label: {df['label'].value_counts().to_dict()}")
df[["text", "text_en", "label"]].head(3)

## 2. Mengukur Kemiripan Vektor

Tiga ukuran yang paling sering dipakai:

| Ukuran | Rumus | Arti |
|---|---|---|
| **Cosine similarity** | $\frac{a \cdot b}{\lVert a \rVert \lVert b \rVert}$ | sudut antara vektor (−1 sampai 1); **mengabaikan panjang** vektor |
| **Dot product** | $a \cdot b$ | sudut **dan** panjang |
| **Jarak euclidean** | $\lVert a - b \rVert$ | jarak garis lurus (kecil = mirip) |

Fakta penting: jika semua vektor **dinormalisasi** (panjang = 1), ketiganya memberi **urutan yang sama**: cosine = dot product, dan $\lVert a-b \rVert^2 = 2 - 2\cos$. Karena itu embedding biasanya dinormalisasi, lalu pencarian cukup memakai dot product yang paling cepat dihitung.

In [ ]:
import matplotlib.pyplot as plt

a, b, c_ = np.array([3.0, 1.0]), np.array([6.0, 2.2]), np.array([1.0, 3.0])

def cos(x, y):
    return x @ y / (np.linalg.norm(x) * np.linalg.norm(y))

for nama, v in [("b (arah sama, lebih panjang)", b), ("c (arah berbeda)", c_)]:
    print(f"a vs {nama:<28} cosine {cos(a, v):.3f} | dot {a @ v:6.2f} | euclidean {np.linalg.norm(a - v):.2f}")

an, bn = a / np.linalg.norm(a), b / np.linalg.norm(b)
print(f"\nSetelah normalisasi: cosine {cos(an, bn):.4f} = dot {an @ bn:.4f} | euclid² {np.sum((an - bn) ** 2):.4f} = 2−2cos {2 - 2 * cos(an, bn):.4f}")

plt.figure(figsize=(4, 4))
for v, nama, w in [(a, "a", "tab:blue"), (b, "b", "tab:green"), (c_, "c", "tab:red")]:
    plt.arrow(0, 0, *v, head_width=0.15, color=w, length_includes_head=True); plt.text(*v, f" {nama}", color=w)
plt.xlim(-0.5, 7); plt.ylim(-0.5, 4); plt.gca().set_aspect("equal"); plt.title("cosine melihat arah, bukan panjang"); plt.show()

## 3. Model Embedding Kalimat

Model sentence-transformers dilatih (dengan *contrastive learning*, mirip CLIP di modul 05) agar kalimat yang maknanya sama menghasilkan vektor yang dekat, **walaupun kata-katanya berbeda atau bahasanya berbeda**.

`model.encode` menerima daftar kalimat dan mengembalikan matriks `(n_kalimat, 384)`. Dengan `normalize_embeddings=True`, cosine similarity = dot product.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print("dimensi embedding:", model.get_sentence_embedding_dimension(), "| panjang maks:", model.max_seq_length, "token")

kalimat = [
    "Makanannya enak sekali dan porsinya besar.",
    "Rasa masakannya lezat, porsinya juga banyak.",
    "The food was delicious and the portions were generous.",
    "Paketnya datang terlambat tiga hari.",
    "Pengiriman barangnya sangat lama.",
    "Toko buka setiap hari mulai jam delapan pagi.",
]
emb = model.encode(kalimat, normalize_embeddings=True)
sim = emb @ emb.T
print("bentuk embedding:", emb.shape)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6)); ax.set_yticklabels([k[:38] for k in kalimat], fontsize=8)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", color="w" if sim[i, j] < 0.6 else "k", fontsize=8)
plt.colorbar(im); plt.title("Cosine similarity antar kalimat"); plt.show()

Perhatikan: kalimat 1 dan 2 (makna sama, kata berbeda) mirip, **begitu pula kalimat 3 yang berbahasa Inggris**. Kalimat tentang pengiriman membentuk kelompok sendiri.

### 3.1 Di balik layar: mean pooling

Model ini adalah Transformer encoder (seperti BERT, modul 06) yang menghasilkan satu vektor **per token**. Untuk mendapatkan satu vektor **per kalimat**, vektor semua token dirata-ratakan (**mean pooling**), dengan mengabaikan token padding lewat `attention_mask`. Kita lakukan manual dengan `transformers` dan cocokkan hasilnya dengan `model.encode`.

In [ ]:
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
enc_model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

batch = tok(kalimat[:3], padding=True, truncation=True, max_length=model.max_seq_length, return_tensors="pt").to(device)
with torch.no_grad():
    token_emb = enc_model(**batch).last_hidden_state                 # (3, n_token, 384)
mask = batch["attention_mask"].unsqueeze(-1).float()                   # (3, n_token, 1)
manual = (token_emb * mask).sum(1) / mask.sum(1)                        # rata-rata token asli saja

print("embedding per token:", tuple(token_emb.shape), "→ per kalimat:", tuple(manual.shape))
ref = model.encode(kalimat[:3], convert_to_tensor=True).to(device)
print("sama dengan model.encode?", torch.allclose(manual, ref, atol=1e-4))

## 4. Semantic Search vs Pencarian Kata Kunci

Kita encode seluruh 1.000 kalimat bahasa Indonesia **sekali saja**. Pencarian lalu menjadi: encode query → dot product dengan semua embedding → ambil skor tertinggi.

Sebagai pembanding, **TF-IDF** (modul 06) hanya mencocokkan **kata yang sama persis**. Query dengan sinonim atau parafrase akan gagal.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus_emb = model.encode(df["text"].tolist(), normalize_embeddings=True, batch_size=64, show_progress_bar=False)
tfidf = TfidfVectorizer(sublinear_tf=True).fit(df["text"])
corpus_tfidf = tfidf.transform(df["text"])
print("embedding korpus:", corpus_emb.shape)


def cari_semantik(query, k=3):
    q = model.encode([query], normalize_embeddings=True)[0]
    skor = corpus_emb @ q
    idx = np.argsort(-skor)[:k]
    return [(skor[i], df["text"][i]) for i in idx]


def cari_tfidf(query, k=3):
    skor = (corpus_tfidf @ tfidf.transform([query]).T).toarray().ravel()
    idx = np.argsort(-skor)[:k]
    return [(skor[i], df["text"][i]) for i in idx]


for query in ["rasanya hambar dan tidak enak", "kurir lambat mengantar paket", "tempat wisata yang indah"]:
    print(f"\n🔎 {query}")
    for nama, fn in [("semantik", cari_semantik), ("TF-IDF", cari_tfidf)]:
        for s, t in fn(query):
            print(f"   [{nama:<8} {s:.2f}] {t[:110]}")

## 5. Mengukur Kualitas Pencarian

Menilai hasil pencarian "secara feeling" tidak cukup. Kita butuh **ground truth**: untuk setiap query, dokumen mana yang benar?

NusaX yang paralel memberi ground truth gratis: gunakan **kalimat bahasa Inggris sebagai query**, lalu cari di korpus **bahasa Indonesia**. Jawaban benar = kalimat Indonesia dengan `id` yang sama.

Metrik:
- **Recall@k**: persentase query yang jawaban benarnya muncul di k hasil teratas.
- **MRR** (*Mean Reciprocal Rank*): rata-rata `1 / peringkat jawaban benar`. Nilai 1 berarti selalu di peringkat pertama.

Ini adalah metrik standar evaluasi *retrieval*, dan akan dipakai lagi untuk RAG di modul 09.

In [ ]:
def evaluasi(skor_matrix, ks=(1, 5, 10)):
    # skor_matrix[i, j] = skor query i terhadap dokumen j; jawaban benar query i adalah dokumen i
    benar = np.diag(skor_matrix)
    peringkat = (skor_matrix > benar[:, None]).sum(1) + 1          # 1 = paling atas
    hasil = {f"recall@{k}": np.mean(peringkat <= k) for k in ks}
    hasil["MRR"] = np.mean(1 / peringkat)
    return hasil


query_en = df["text_en"].tolist()
skor_emb = model.encode(query_en, normalize_embeddings=True, batch_size=64, show_progress_bar=False) @ corpus_emb.T
skor_tfidf = (tfidf.transform(query_en) @ corpus_tfidf.T).toarray()

hasil = pd.DataFrame({
    "embedding multibahasa": evaluasi(skor_emb),
    "TF-IDF (kata kunci)": evaluasi(skor_tfidf),
}).T
print("Query bahasa Inggris → cari terjemahannya di 1.000 kalimat bahasa Indonesia")
hasil.round(3)

TF-IDF masih bisa menemukan sebagian terjemahan, tetapi hanya lewat kata yang **kebetulan sama** di kedua bahasa: nama merek (`grab`, `gojek`), nama tempat, dan angka. Untuk kalimat tanpa kata seperti itu, TF-IDF buta. Model embedding multibahasa dilatih agar kalimat dan terjemahannya berdekatan, sehingga mampu menemukan pasangan terjemahan berdasarkan **makna**.

Sebaliknya, untuk query yang memakai kata persis seperti dokumen (kode produk, nama orang, istilah teknis), TF-IDF sering unggul. Karena itu sistem produksi biasanya menggabungkan keduanya (**hybrid search**, Latihan 2 dan modul 09).

## 6. Visualisasi Ruang Embedding

Embedding 384 dimensi kita proyeksikan ke 2 dimensi dengan **PCA** (reduksi dimensi linear dari ML dasar). Warna = label sentimen. Model ini tidak dilatih khusus untuk sentimen, jadi pemisahannya tidak sempurna, tetapi pola kelompok biasanya tetap terlihat.

In [ ]:
from sklearn.decomposition import PCA

xy = PCA(n_components=2, random_state=0).fit_transform(corpus_emb)
warna = {"negative": "tab:red", "neutral": "tab:gray", "positive": "tab:green"}
plt.figure(figsize=(7, 6))
for lab, w in warna.items():
    m = (df["label"] == lab).to_numpy()
    plt.scatter(xy[m, 0], xy[m, 1], s=8, alpha=0.6, c=w, label=lab)
plt.legend(); plt.title("Embedding 1.000 kalimat NusaX (PCA 2D)"); plt.show()

## Ringkasan

- Embedding = vektor makna; kemiripan diukur dengan **cosine** (setelah normalisasi = dot product).
- Model sentence-transformers = Transformer encoder + **mean pooling**, dilatih secara contrastive agar parafrase berdekatan, bahkan lintas bahasa.
- **Semantic search** menangkap sinonim dan parafrase; **TF-IDF** unggul untuk pencocokan kata persis. Keduanya saling melengkapi.
- Ukur kualitas pencarian dengan **recall@k** dan **MRR** memakai ground truth.

## Latihan

1. **Bandingkan model.** Ulangi evaluasi bagian 5 dengan model lain, misalnya `sentence-transformers/LaBSE` (lebih besar) atau `intfloat/multilingual-e5-small` (perhatikan: model E5 meminta awalan `"query: "` dan `"passage: "`). Periksa lisensinya, lalu bandingkan recall@1, MRR, dimensi vektor, dan waktu encode dalam satu tabel.
2. **Hybrid search & bahasa daerah.** NusaX juga tersedia dalam bahasa Jawa, Sunda, dan bahasa daerah lain (folder `javanese`, `sundanese`, … di repo yang sama). Ulangi evaluasi bagian 5 dengan query berbahasa Jawa dan Sunda. Lalu gabungkan skor: `skor = α·semantik + (1−α)·tfidf` (normalisasi keduanya ke 0–1) dan cari α terbaik untuk tiap bahasa. Kapan hybrid mengalahkan keduanya?
3. **Duplikat & klaster.** Cari pasangan kalimat di korpus dengan cosine > 0.9 (kandidat duplikat), lalu kelompokkan 1.000 embedding dengan K-Means (k = 8). Baca 5 kalimat dari tiap klaster dan beri nama topiknya. Apakah klaster mengikuti sentimen atau topik (makanan, transportasi, dan sebagainya)?